# Around the World in Eighty Days - Geoparsing Demo

This notebook demonstrates the capabilities of the Irchel Geoparser by mapping populated places mentioned in Jules Verne's classic novel "Around the World in Eighty Days". We will:

1. Download the book text from Project Gutenberg
2. Extract and clean the chapters
3. Use the Irchel Geoparser to identify and resolve place names
4. Create an interactive map displaying mentioned places with context

## Setup and Imports

In [ ]:
import re
import requests
from collections import defaultdict

from geoparser import Geoparser
from geoparser.modules import SpacyRecognizer, SentenceTransformerResolver

import plotly.graph_objects as go

## Download and Parse the Book

In [ ]:
# Download the book from Project Gutenberg
url = "https://www.gutenberg.org/files/103/103-0.txt"
response = requests.get(url)
response.encoding = "utf-8"
full_text = response.text

print(f"Downloaded {len(full_text)} characters")

In [ ]:
# Split into lines for processing
lines = full_text.split("\n")

# Find the start of Chapter I (the actual content)
start_idx = None
for i, line in enumerate(lines):
    if line.strip().startswith("CHAPTER I."):
        start_idx = i
        break

# Find the end marker
end_idx = None
for i, line in enumerate(lines):
    if "*** END OF THE PROJECT GUTENBERG EBOOK" in line:
        end_idx = i
        break

# Extract only the book content
book_lines = lines[start_idx:end_idx]
book_text = "\n".join(book_lines)

print(f"Extracted {len(book_text)} characters of book content")

## Extract Chapters

In [ ]:
# Function to convert Roman numerals to integers
def roman_to_int(roman):
    """Convert Roman numeral to integer."""
    roman_values = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    total = 0
    prev_value = 0

    for char in reversed(roman):
        value = roman_values[char]
        if value < prev_value:
            total -= value
        else:
            total += value
        prev_value = value

    return total


# Find all chapter boundaries
chapter_pattern = re.compile(r"^CHAPTER ([IVXLCDM]+)\.\s*$", re.MULTILINE)
chapters = []

matches = list(chapter_pattern.finditer(book_text))

for i, match in enumerate(matches):
    chapter_num = roman_to_int(match.group(1))
    start_pos = match.start()

    # End position is start of next chapter or end of book
    if i < len(matches) - 1:
        end_pos = matches[i + 1].start()
    else:
        end_pos = len(book_text)

    chapter_text = book_text[start_pos:end_pos]
    chapters.append({"number": chapter_num, "text": chapter_text, "start": start_pos})

print(f"Extracted {len(chapters)} chapters")

## Initialize the Geoparser

We'll use the transformer-based spaCy model (`en_core_web_trf`) for more accurate toponym recognition, combined with the sentence transformer resolver for toponym resolution.

In [ ]:
# Create a recognizer using the transformer-based spaCy model
recognizer = SpacyRecognizer(model_name="en_core_web_trf")

# Create a resolver using sentence transformers
# min_similarity=0.7 provides higher precision at the cost of recall
resolver = SentenceTransformerResolver(min_similarity=0.7)

# Initialize the geoparser with our custom modules
geoparser = Geoparser(recognizer=recognizer, resolver=resolver)

print("Geoparser initialized with:")
print(f"  Recognizer: {recognizer}")
print(f"  Resolver: {resolver}")

## Geoparse the Text

Now we'll use the Irchel Geoparser to identify place names and link them to geographic locations.

In [ ]:
# Parse each chapter
print("Geoparsing chapters (this may take a few minutes)...")
chapter_texts = [ch["text"] for ch in chapters]
documents = geoparser.parse(chapter_texts)

print(f"Processed {len(documents)} documents")

## Extract and Filter Toponyms

We'll filter for populated places (feature_class='P') and collect all mentions with their context.

In [ ]:
# Collect all toponyms by location, filtering for populated places
location_data = defaultdict(lambda: {"mentions": [], "location": None, "count": 0})

for doc_idx, doc in enumerate(documents):
    chapter_num = chapters[doc_idx]["number"]

    for toponym in doc.toponyms:
        # Only include resolved toponyms that are populated places
        if toponym.location is None:
            continue

        feature_class = toponym.location.data.get("feature_class")
        if feature_class != "P":
            continue

        # Get location identifier (using geonames_id if available)
        loc_id = toponym.location.data.get("geonames_id") or toponym.location.data.get(
            "name"
        )

        # Extract context (30 chars before and after)
        start = max(0, toponym.start - 30)
        end = min(len(doc.text), toponym.end + 30)
        context = doc.text[start:end]

        # Clean up context
        context = context.replace("\n", " ").strip()

        # Make the toponym bold in the context
        toponym_text = toponym.text
        context_with_bold = context.replace(toponym_text, f"<b>{toponym_text}</b>")

        # Add ellipsis if we truncated
        if start > 0:
            context_with_bold = "..." + context_with_bold
        if end < len(doc.text):
            context_with_bold = context_with_bold + "..."

        location_data[loc_id]["mentions"].append(
            {
                "chapter": chapter_num,
                "context": context_with_bold,
                "toponym_text": toponym_text,
            }
        )
        location_data[loc_id]["location"] = toponym.location
        location_data[loc_id]["count"] += 1

print(f"Found {len(location_data)} unique populated places")
print(f"Total mentions: {sum(data['count'] for data in location_data.values())}")

In [ ]:
# Show top 10 most mentioned places
sorted_locations = sorted(
    location_data.items(), key=lambda x: x[1]["count"], reverse=True
)
print("\nTop 10 most mentioned places:")
for loc_id, data in sorted_locations[:10]:
    name = data["location"].data.get("name")
    country = data["location"].data.get("country_name")
    count = data["count"]
    print(f"  {name}, {country}: {count} mentions")

## Create Interactive Map

Now we'll create a Plotly map showing all the locations with interactive hover information.

In [ ]:
# Prepare data for plotting
lats = []
lons = []
names = []
sizes = []
hover_texts = []

# Maximum number of lines in hover popup to prevent rendering issues
MAX_HOVER_LINES = 25

for loc_id, data in location_data.items():
    loc = data["location"]

    # Get coordinates
    lat = loc.data.get("latitude")
    lon = loc.data.get("longitude")

    if lat is None or lon is None:
        continue

    lats.append(lat)
    lons.append(lon)

    # Build location name hierarchy
    name_parts = []
    for field in ["name", "admin2_name", "admin1_name", "country_name"]:
        value = loc.data.get(field)
        if value and value not in name_parts:
            name_parts.append(value)

    location_name = ", ".join(name_parts)
    names.append(location_name)

    # Set marker size
    count = data["count"]
    size = count + 8
    sizes.append(size)

    # Build hover text with line limit
    hover_lines = [
        f"<b>{location_name}</b>",
        f"<i>Mentioned {count} time{'s' if count > 1 else ''}</i>",
    ]

    # Group mentions by chapter
    mentions_by_chapter = defaultdict(list)
    for mention in data["mentions"]:
        mentions_by_chapter[mention["chapter"]].append(mention["context"])

    # Count lines we'll use (2 for header, 2 per chapter title, 1 per mention)
    current_lines = 2
    included_chapters = 0

    # Count how many mentions we actually show
    displayed_mentions = 0
    total_mentions = sum(len(v) for v in mentions_by_chapter.values())

    # Add mentions with line limit
    for chapter_num in sorted(mentions_by_chapter.keys()):
        contexts = mentions_by_chapter[chapter_num]

        # Prevent adding a chapter header unless at least one context fits
        if current_lines + 3 > MAX_HOVER_LINES:
            break

        hover_lines.append(f"<br><b>Chapter {chapter_num}:</b>")
        current_lines += 2
        included_chapters += 1

        # Add contexts
        first_context_added = False
        for context in contexts:
            if current_lines + 1 > MAX_HOVER_LINES:
                break
            hover_lines.append(f"    {context}")
            current_lines += 1
            first_context_added = True
            displayed_mentions += 1

        # If no contexts fit, remove chapter header
        if not first_context_added:
            hover_lines.pop()
            current_lines -= 2
            included_chapters -= 1
            break

    # Add indicator if there are more mentions
    if displayed_mentions < total_mentions:
        remaining_mentions = total_mentions - displayed_mentions
        remaining_chapters = len(mentions_by_chapter) - included_chapters
        hover_lines.append(
            f"<br><i>... and {remaining_mentions} more mention{'s' if remaining_mentions > 1 else ''} in {remaining_chapters} other chapter{'s' if remaining_chapters > 1 else ''}</i>"
        )

    hover_texts.append("<br>".join(hover_lines))

print(f"Prepared {len(lats)} locations for mapping")

In [ ]:
# Create the Plotly figure
fig = go.Figure(
    data=go.Scattergeo(
        lon=lons,
        lat=lats,
        text=names,
        mode="markers",
        marker=dict(
            size=sizes,
            color="steelblue",
            line=dict(width=0.5, color="black"),
            sizemode="diameter",
            opacity=0.5,
        ),
        hovertext=hover_texts,
        hoverinfo="text",
        hoverlabel=dict(
            bgcolor="white", font_size=12, font_family="Arial", align="left"
        ),
    )
)

fig.update_layout(
    title=dict(
        text='Place Names Mentioned in "Around the World in Eighty Days" by Jules Verne<br><sub>Marker size indicates mention frequency</sub>',
        x=0.5,
        xanchor="center",
    ),
    geo=dict(
        projection_type="natural earth",
        showland=True,
        landcolor="rgb(243, 243, 243)",
        coastlinecolor="rgb(204, 204, 204)",
        showcountries=True,
        countrycolor="rgb(204, 204, 204)",
    ),
    height=500,
    margin=dict(l=0, r=0, t=80, b=0),
)

# Display the map
fig.show()